In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import numpy as np
from time import sleep
#from scipy.spatial.transform import Rotation as R

from URBasic import Joint6D
from URBasic import TCP6D
from URBasic import TCP6DDescriptor

import src.calibration as cal
import src.space as sp

SIMULATION = True
# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
robot_ip = "10.30.5.159"
robot_opened_gripper_size_mm = 40. # Max oppenning of gripper

# Base robot position in Joint space
j_o = Joint6D.createFromRadians(
    -1.5708, -1.5708, -1.0472, -2.0944, 1.5708, 0.0
)

In [2]:
# Simulator (swap just this line)
if SIMULATION:
    from my_simulation import ISCoinSim as ISCoin
    robot_ip = ""

    iscoin = ISCoin(opened_gripper_size_mm=robot_opened_gripper_size_mm)
else:
    from URBasic import ISCoin

    iscoin = ISCoin(robot_ip, robot_opened_gripper_size_mm)

    if not iscoin.isInitOk:
        print("Error robot not initialised")
        sys.exit(1)

    iscoin.robot_control.reset_error()

ISCoinSim connected to container 'iscoin_simulator'


In [3]:
robot_arm = iscoin.robot_control

In [4]:
if not SIMULATION:
    tcps = cal.collect_data(robot_arm, 20)
else:
    tcps = [
        [0.0024946337740892888, -0.3594913915339868, 0.06033167700558813, -0.62069755529755, -2.369727486212462, -0.13752393611863362],
        [-0.07482225018547103, -0.35685269130655106, 0.07009337653443365, 1.9372057315922462, 2.190232377619723, 0.6719840297817087],
        [-0.08776793211801745, -0.2979812165503483, 0.051154992555969794, 2.3781388566898687, 0.2623998383341909, 0.7178051284770465],
        [-0.0006933205535983589, -0.340924257935631, 0.06551204074775754, -2.879088789274111, -0.4716877213448937, 0.8276834472055694],
        [-0.04094004360899719, -0.3437262194580261, 0.0786140415986546, -3.087426800813378, 0.2627406174702693, 0.06421242457662117],
        [0.026621421400990636, -0.3469050147213359, 0.033956680032955974, -2.5014091646201537, -0.22918589994059693, 1.5168173293250897],
    ]

In [5]:
tcp_offset = cal.get_tcp_offset(tcps)
if not SIMULATION:
    robot_arm.set_tcp(tcp_offset)
    robot_arm.freedrive_mode()
    tcp_ref = None
    while not tcp_ref:
        i = input(f"Confirm reference position for TCP validation. y/n")
        if i.lower()[0] == "y":
            robot_arm.end_freedrive_mode()
            joint_ref = robot_arm.get_actual_joint_positions()

    cal.validate_calibration(robot_arm, joint_ref)

TCP consistency (mm):    	 0.5873600179479618
Flange motion (cm):      	 10.961315598701008
Rotation variation (deg):	 36.9973854515488
TCP in flange frame: [-0.00072928 -0.00040785  0.07966732]  <-
TCP in tool   frame: [-0.04418502 -0.33927468 -0.00117747]


In [6]:
if not SIMULATION:
    p_world = np.array([
        #TODO:  Give the measure points of the object
        #       Here the plaque.png
        [-5.5, -19.5, -2],
        [-5.5, 105.5, -2],
        [69.5, 105.5, -2],
        [69.5, -19.5, -2],
    ])/1000
    p_tcp = sp.collect_data(robot_arm, p_world)
else:
    p_world = np.array([
        [-5.5, -19.5, -2],
        [-5.5, 105.5, -2],
        [69.5, 105.5, -2],
        [69.5, -19.5, -2],
    ])/1000

    p_tcp = np.array([
        [-0.05, -0.2, 0.2500, 0, 3.14, 0],
        [-0.05, -0.4, 0.2500, 0, 3.14, 0],
        [ 0.05, -0.4, 0.2500, 0, 3.14, 0],
        [ 0.05, -0.2, 0.2500, 0, 3.14, 0],
    ])

In [7]:
import numpy as np

def affine_transform_rot_3D(p_world, p_tcp):
    """
    Compute rigid transform (R, t) such that:
        p_tcp ≈ R * p_world + t
    using the Kabsch algorithm.
    Returns a function that transforms a point and a normal vector.
    """

    # Extract only XYZ from p_tcp (ignore rx, ry, rz)
    p_tcp_xyz = p_tcp[:, :3]

    # Compute centroids
    c_world = p_world.mean(axis=0)
    c_tcp   = p_tcp_xyz.mean(axis=0)

    # Center the points
    W = p_world - c_world
    T = p_tcp_xyz - c_tcp

    # Compute covariance matrix
    H = W.T @ T

    # SVD for rotation
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # Fix improper rotation (reflection)
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    # Translation
    t = c_tcp - R @ c_world

    # Return a function that applies the transform
    def transform(point, normal):
        point = np.array(point)
        normal = np.array(normal)

        # Transform point
        p_new = R @ point + t

        # Rotate normal
        z_new = R @ normal
        z_new = z_new / np.linalg.norm(z_new)

        # Build a full orthonormal frame
        # Choose an arbitrary vector not parallel to z_new
        tmp = np.array([1, 0, 0]) if abs(z_new[0]) < 0.9 else np.array([0, 1, 0])

        x_new = np.cross(tmp, z_new)
        x_new /= np.linalg.norm(x_new)

        y_new = np.cross(z_new, x_new)

        # Rotation matrix
        R_new = np.column_stack((x_new, y_new, z_new))

        # Convert to RPY (XYZ convention)
        ry = np.arcsin(-R_new[2,0])
        rx = np.arctan2(R_new[2,1], R_new[2,2])
        rz = np.arctan2(R_new[1,0], R_new[0,0])

        return (*p_new, rx, ry, rz)

    return transform


In [8]:
p_world = np.array([
    [-5.5, -19.5, -2],
    [-5.5, 105.5, -2],
    [69.5, 105.5, -2],
    [69.5, -19.5, -2],
])/1000
rx = 0
ry = 0
rz = 0
p_tcp = np.array([
    [-0.05, -0.2, 0.2500, rx, ry, rz],
    [-0.05, -0.4, 0.2500, rx, ry, rz],
    [ 0.05, -0.4, 0.2500, rx, ry, rz],
    [ 0.05, -0.2, 0.2500, rx, ry, rz],
])
world_to_tcp = affine_transform_rot_3D(p_world, p_tcp)

center      = [0.032, 0.046, 0]
normal      = [1,     0,  0]
x,y,z,rx,ry,rz = world_to_tcp(center, normal)
print(x,y,z,rx,ry,rz)

0.0 -0.303 0.248 0.0 1.5707963267948966 0.0


In [9]:
import src.draw as draw

center      = [0.032, 0.046, 0]
path = draw.draw_cercle(center, 0.03, 4)

In [10]:
normal      = [1,     0,  0]
x,y,z,rx,ry,rz = world_to_tcp(center, normal)
print(x,y,z,rx,ry,rz)
normal      = [0,     0,  1]
x,y,z,rx,ry,rz = world_to_tcp(center, normal)
print(x,y,z,rx,ry,rz)

0.0 -0.303 0.248 0.0 1.5707963267948966 0.0
0.0 -0.303 0.248 3.141592653589793 -0.0 1.5707963267948966


In [ ]:
normals = [
    [ 1,  0,  0],
    [-1,  0,  0],
    [ 0,  1,  0],
    [ 0, -1,  0],
    [ 0,  0,  1],
    [ 0,  0, -1],
]
motions = []
for n in normals:
    motion = []
    normal = n
    for p in path:
        x,y,z,rx,ry,rz = world_to_tcp(p, normal)
        t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
        m = TCP6DDescriptor(t, a=5 ,v=1, t=0)
        motion.append(m)
    print(motion[0].tcp.toList())
    motions.append(motion)

In [13]:
robot_arm.movej(j_o)

movej sent (duration=3s)
Movement OK — target reached


True

In [ ]:
for motion in motions:
    #robot_arm.movej(j_o)
    robot_arm.movel_waypoints(motion)
    robot_arm.movej(j_o)

# Test draw

In [ ]:
robot_arm.get_actual_tcp_pose()

In [ ]:
def normal_to_rxyz(n):
    x,y,z = n
    z = -z
    rx = np.arctan2(-y, np.sqrt(x**2 + z**2))
    ry = np.arctan2(x, z)
    rz = 0
    return [rx,ry,rz]

def affine_transform_rot_3D(p_world, p_tcp):
    """
    Compute rigid transform (R, t) such that:
        p_tcp ≈ R * p_world + t
    using the Kabsch algorithm.
    Returns a function that transforms a point and a normal vector.
    """

    # Extract only XYZ from p_tcp (ignore rx, ry, rz)
    p_tcp_xyz = p_tcp[:, :3]

    # Compute centroids
    c_world = p_world.mean(axis=0)
    c_tcp   = p_tcp_xyz.mean(axis=0)

    # Center the points
    W = p_world - c_world
    T = p_tcp_xyz - c_tcp

    # Compute covariance matrix
    H = W.T @ T

    # SVD for rotation
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # Fix improper rotation (reflection)
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    # Translation
    t = c_tcp - R @ c_world

    # Return a function that applies the transform
    def transform(point, normal):
        point = np.array(point)
        normal = np.array(normal)

        # Transform cartesian
        p_new = R @ point + t

        # Transform rotation
        r_new = normal_to_rxyz(normal)
        n_new = R @ r_new

        return (*p_new, *n_new)

    return transform


In [ ]:
p_world = np.array([
    [-5.5, -19.5, -2],
    [-5.5, 105.5, -2],
    [69.5, 105.5, -2],
    [69.5, -19.5, -2],
])/1000
rx = 0
ry = 0
rz = 0
p_tcp = np.array([
    [-0.05, -0.2, 0.2500, rx, ry, rz],
    [-0.05, -0.4, 0.2500, rx, ry, rz],
    [ 0.05, -0.4, 0.2500, rx, ry, rz],
    [ 0.05, -0.2, 0.2500, rx, ry, rz],
])
world_to_tcp = affine_transform_rot_3D(p_world, p_tcp)

center      = [0.032, 0.046, 0]
normal      = [1,     0,  0]
x,y,z,rx,ry,rz = world_to_tcp(center, normal)
print(x,y,z,rx,ry,rz)

In [ ]:
normals = [
    [ 1,  0,  0],
    [-1,  0,  0],
    [ 0,  1,  0],
    [ 0, -1,  0],
    [ 0,  0,  1],
    [ 0,  0, -1],
]
motions = []
for n in normals:
    x,y,z,rx,ry,rz = world_to_tcp(center, n)
    print(x,y,z,rx,ry,rz)

In [ ]:
normals = [
    [ 1,  0,  0],
    [-1,  0,  0],
    [ 0,  1,  0],
    [ 0, -1,  0],
    [ 0,  0,  1], # <- Plane
    [ 0,  0, -1],
]
motions = []
for n in normals:
    motion = []
    normal = n
    for p in path:
        x,y,z,rx,ry,rz = world_to_tcp(p, normal)
        print(rx,ry,rz)
        t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
        m = TCP6DDescriptor(t, a=5 ,v=1, t=0)
        motion.append(m)
    print(normal)
    print(motion[0].tcp.toList())
    motions.append(motion)

In [ ]:
for i, motion in enumerate(motions):
    print(normals[i])
    #robot_arm.movej(j_o)
    robot_arm.movel_waypoints(motion)
    robot_arm.movej(j_o)

In [ ]:
def rotvec_to_rotmat(r):
    """Convert rotation vector to rotation matrix using Rodrigues' formula."""
    theta = np.linalg.norm(r)
    if theta < 1e-12:
        return np.eye(3)
    k = r / theta
    K = np.array([[0, -k[2], k[1]],
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])
    R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)
    return R

def rotmat_to_rotvec(R):
    """Convert rotation matrix to rotation vector."""
    theta = np.arccos((np.trace(R) - 1) / 2)
    if theta < 1e-12:
        return np.zeros(3)
    rx = (R[2,1] - R[1,2]) / (2*np.sin(theta))
    ry = (R[0,2] - R[2,0]) / (2*np.sin(theta))
    rz = (R[1,0] - R[0,1]) / (2*np.sin(theta))
    return theta * np.array([rx, ry, rz])

def pose_to_matrix(p):
    """Convert UR pose to 4x4 homogeneous matrix."""
    x, y, z, rx, ry, rz = p
    R = rotvec_to_rotmat(np.array([rx, ry, rz]))
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = [x, y, z]
    return T

def matrix_to_pose(T):
    """Convert 4x4 homogeneous matrix back to UR pose."""
    x, y, z = T[:3, 3]
    R = T[:3, :3]
    rx, ry, rz = rotmat_to_rotvec(R)
    return [x, y, z, rx, ry, rz]

def pose_trans(p1, p2):
    """Equivalent of UR's pose_trans(p1, p2)."""
    T1 = pose_to_matrix(p1)
    T2 = pose_to_matrix(p2)
    T = T1 @ T2
    x, y, z, rx, ry, rz = matrix_to_pose(T)
    return x, y, z, rx, ry, rz


In [ ]:
tcp_test = [0.02,.04,.05,1,0,1.56]
tcp_trans = [0,0,0.05,0,0,0]
tcp_result = pose_trans(tcp_test,tcp_trans)
print(tcp_test)
print(tcp_trans)
print(tcp_result)

In [ ]:
robot_arm.get_actual_tcp_pose()